In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
import joblib
import os
from datetime import datetime

In [2]:
#modular pipeline architecture for heart disease predication
print("=" * 80)
print("Heart Disease Predication - Modular Pipeline Architecture")
print("=" * 80)

Heart Disease Predication - Modular Pipeline Architecture


In [ ]:
#1. Load dataset
print("\n" + "=" * 80)
print("1. data loading")
print("=" * 80)

# Load the engineered dataset (or create a sample if not available)
try:
    df = pd.read_csv("../Data/processed/heart_disease_engineered.csv")
    print("loaded engineered dataset")
except:
    # Create sample data for demonstration
    print("Engineered dataset not found, creating sample data")
    np.random.seed(42)
    n_samples = 1000

print(f"dataset shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")


1. data loading
Engineered dataset not found, creating sample data


In [ ]:
# Feature Categorisation
print("\n" + "=" * 80)
print("2. feature categorization")
print("=" * 80)

# exclude identifiers and target
excluded_cols = ['patient_id', 'heart_disease_present']

# define feature groups based on your project analysis
numerical_features = [
    'age', 'resting_blood_pressure', 'serum_cholesterol_mg_per_dl',
    'max_heart_rate_achieved', 'oldpeak_eq_st_depression', 'num_major_vessels'
]

categorical_features = [
    'thal', 'chest_pain_type', 'resting_ekg_results',
    'slope_of_peak_exercise_st_segment'
]

binary_features = [
    'sex', 'exercise_induced_angina', 'fasting_blood_sugar_gt_120_mg_per_dl'
]

# identify engineered features (any features not in the above categories)
all_features = [col for col in df.columns if col not in excluded_cols]
engineered_features = [
    col for col in all_features 
    if col not in numerical_features + categorical_features + binary_features
]

print(f"numerical features ({len(numerical_features)}): {numerical_features}")
print(f"categorical features ({len(categorical_features)}): {categorical_features}")
print(f"binary features ({len(binary_features)}): {binary_features}")
print(f"engineered features ({len(engineered_features)}): {engineered_features}")


In [ ]:
#3. create modular pipepline
print("\n" + "=" * 80)
print("3. create modular pipelines")
print("=" * 80)

# numerical pipeline
print("\nCreating a numerical Pipeline")
# define which scaler to use for each numerical feature (based on your scaling strategy)
# this should come from your scaling documentation
numerical_scalers = {
    'age': 'minmax',
    'resting_blood_pressure': 'standard',
    'serum_cholesterol_mg_per_dl': 'standard',
    'max_heart_rate_achieved': 'standard',
    'oldpeak_eq_st_depression': 'robust',
    'num_major_vessels': 'minmax'
}

# separate features by scaler type
standard_features = [f for f in numerical_features if numerical_scalers.get(f) == 'standard']
minmax_features = [f for f in numerical_features if numerical_scalers.get(f) == 'minmax']
robust_features = [f for f in numerical_features if numerical_scalers.get(f) == 'robust']

print(f"standardscaler: {standard_features}")
print(f"minmaxscaler: {minmax_features}")
print(f"robustscaler: {robust_features}")

# create numerical pipeline with different scalers
numerical_pipeline = ColumnTransformer([
    ('standard', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), standard_features),
    
    ('minmax', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]), minmax_features),
    
    ('robust', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ]), robust_features)
])

In [ ]:
# Categorical feature pipeline
print("\nCreating categorical pipeline...")

# define encoding strategies for each categorical feature
categorical_encoding = {
    'thal': 'onehot',  # no ordinal relationship
    'chest_pain_type': 'ordinal',  # has severity order
    'resting_ekg_results': 'onehot',  # categorical
    'slope_of_peak_exercise_st_segment': 'ordinal'  # has order (1=upsloping, 2=flat, 3=downsloping)
}

# separate features by encoding type
onehot_features = [f for f in categorical_features if categorical_encoding.get(f) == 'onehot']
ordinal_features = [f for f in categorical_features if categorical_encoding.get(f) == 'ordinal']

print(f"  onehot encoding: {onehot_features}")
print(f"  ordinal encoding: {ordinal_features}")

# define categories for ordinal encoding (based on your medical knowledge)
ordinal_categories = {
    'chest_pain_type': [[1, 2, 3, 4]],  # 1=typical angina (least severe) to 4=asymptomatic (most severe)
    'slope_of_peak_exercise_st_segment': [[1, 2, 3]]  # 1=upsloping, 2=flat, 3=downsloping (worst)
}

# create categorical pipeline
categorical_pipeline = ColumnTransformer([
    ('onehot', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]), onehot_features),
    
    ('ordinal', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=[
            ordinal_categories.get(feat, sorted(df[feat].unique().tolist())) 
            for feat in ordinal_features
        ]))
    ]), ordinal_features)
])

In [ ]:
#Binary pipeline
print("\nCreating binary pipeline...")

# binary features typically need minimal processing
# just ensure they're 0/1 and handle any missing values
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # optional: could add a scaler if needed, but usually binary stays as-is
])

In [ ]:
#Engineered Features pipeline
print("\nCreating engineered features pipeline...")

# engineered features need different handling based on their type
# first, categorize engineered features
if engineered_features:
    engineered_numerical = [f for f in engineered_features if pd.api.types.is_numeric_dtype(df[f])]
    engineered_categorical = [f for f in engineered_features if not pd.api.types.is_numeric_dtype(df[f])]
    
    print(f"  numerical engineered: {engineered_numerical}")
    print(f"  categorical engineered: {engineered_categorical}")
    
    # create pipeline for engineered features
    engineered_pipeline = ColumnTransformer([
        ('engineered_numerical', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())  # default for engineered numerical
        ]), engineered_numerical),
        
        ('engineered_categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), engineered_categorical)
    ])
else:
    print("  no engineered features found")
    engineered_pipeline = 'passthrough'

In [ ]:
# Combine into master pipeline
print("\n" + "=" * 80)
print("4. combine into master pipeline")
print("=" * 80)

# create the master column transformer
master_transformer = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features),
    ('binary', binary_pipeline, binary_features),
    ('engineered', engineered_pipeline, engineered_features) if engineered_features else ('passthrough', 'passthrough', [])
], remainder='drop')  # drop any unprocessed columns

print("✅ created master column transformer")
print(f"total feature groups: 4")
print(f"total features processed: {len(numerical_features + categorical_features + binary_features + engineered_features)}")

In [ ]:
# Add feature selection and dimensionality reduction
print("\n" + "=" * 80)
print("5. add feature selection and dimensionality reduction")
print("=" * 80)

# create the complete preprocessing pipeline
preprocessing_pipeline = Pipeline([
    ('transformer', master_transformer),
    ('feature_selection', SelectKBest(score_func=f_classif, k='all')),  # will be tuned
    ('dimensionality_reduction', PCA(n_components=0.95))  # keep 95% variance
])

print("✅ created complete preprocessing pipeline")
print("pipeline steps:")
print("  1. column transformer (feature-specific processing)")
print("  2. feature selection (selectkbest with f-classif)")
print("  3. dimensionality reduction (pca keeping 95% variance)")

In [ ]:
# Create model pipelines
print("\n" + "=" * 80)
print("6. create model pipelines")
print("=" * 80)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# model 1: logistic regression
logistic_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

# model 2: random forest
random_forest_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        class_weight='balanced',
        n_jobs=-1
    ))
])

# model 3: support vector machine
svm_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', SVC(
        random_state=42,
        probability=True,
        class_weight='balanced'
    ))
])

# model 4: xgboost
xgb_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('classifier', XGBClassifier(
        random_state=42,
        n_estimators=100,
        use_label_encoder=False,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

print("created 4 model pipelines:")
print("  1. logistic regression (interpretable baseline)")
print("  2. random forest (handles non-linearity)")
print("  3. support vector machine (good for high dimensions)")
print("  4. xgboost (state-of-the-art for tabular data)")